#### PREPROCESSING PIPELINE

### Preprocessing/Feature engineering Step

As of the insights and understanding from the EDA of the "Ticket Description" columns, the preprocessing steps for the columns are,

* Case handlings (Upper case -> Lower case) to make the text in standards to ensure vocabulary consistency. This reduces vocabulary size and prevents artificial feature duplication during vectorization.
* Expand the common contractions of words to make the words in common standards, it's won't removed as noise and make more meanings to embeddings.
* Replace {product_purchased} with the NER token to make the embeddings and model to understand it's named entity (products) which capture the sematics and won't noise the vectors. Substituting actual product names would bias clusters toward product identity rather than issue type. A generic token preserves sentence structure and semantic context while remaining generalizable to unseen data.
* Regex identification of emails, URLs, zips,
handles, numbers which biases the vectors because it carries no semantic meaning related to issue type but introduce arbitrary features that distort vector representations so remove them to make the vectors generalizable.
* Tokenization of the description to preprocessed text into individual word and subword units. Word-level tokenization splits on whitespace and punctuation boundaries, while subword tokenization decomposes domain-specific technical terms such as compound or morphologically complex words to preserve root meaning and improve vocabulary coverage.
* Stopwords removal (with/without custom words) to eliminate high-frequency function word that appear uniformly across all tickets and carry no discriminative power for clustering.   
* Lemmatization of tokens to reduce inflected word forms to their dictionary base form using part-of-speech context.
* Vectorization/embeddings Convert preprocessed tokens into numerical representations for clustering.






In [23]:
# Import the necessary modules
import numpy as np
import scipy.sparse as sp
import sys
import os 
import pandas as pd 

target_dir = r"D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43"
if (os.getcwd() != target_dir):
    os.chdir(target_dir)
    print(f"Directory changed to: {os.getcwd()}")

root_path = os.getcwd()
reports_dir = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'reports')
figures_dir = os.path.join(reports_dir, 'figures')
tables_dir = os.path.join(reports_dir, 'tables')
os.makedirs(figures_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)
print(f"Figures will be saved to: {figures_dir}")
print(f"Tables will be saved to: {tables_dir}")

Figures will be saved to: D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\reports\figures
Tables will be saved to: D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\reports\tables


In [13]:
if root_path not in sys.path:
    sys.path.insert(0, root_path)

# Verify 'src' is now visible
print(f"Project Root: {root_path}")
print(f"Is 'src' folder accessible? {os.path.exists(os.path.join(root_path, 'src'))}")
print(f"Is 'data' folder accessible? {os.path.exists(os.path.join(root_path, 'data'))}")

csv_path = os.path.join(root_path, 'data', 'raw', 'customer_support_tickets.csv')

print(csv_path)

Project Root: D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43
Is 'src' folder accessible? True
Is 'data' folder accessible? True
D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\raw\customer_support_tickets.csv


In [14]:
# Import the necessary modules
from src.pipeline.Preprocessing import to_lower, expand_contractions, replace_product_purchased, remove_noise, tokenize_text, remove_stopwords_all, filter_short_tokens, lemmatize_tokens
from src.features.Embeddings import vectorize_boa, vectorize_tfidf, vectorize_skipgram, vectorize_sbert
from src.utils.Utils import store_dataset

In [15]:
# load the dataset
print(csv_path)

df = pd.read_csv(csv_path)
df.head()

D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\raw\customer_support_tickets.csv


,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [16]:
# Preprocess the text to store the tokens embeddings 
def preprocess_text(text_description: pd.Series):
    text = text_description.copy()
    text = text.apply(to_lower)
    text = text.apply(expand_contractions)
    text = text.apply(replace_product_purchased)
    text = text.apply(remove_noise)
    sbert_text = text.copy()
    tokens = text.apply(tokenize_text)
    tokens = tokens.apply(remove_stopwords_all)
    tokens = tokens.apply(filter_short_tokens)
    tokens = tokens.apply(lemmatize_tokens)
    return tokens, sbert_text

In [17]:
# Invoke the method to preprocess the text descriptions and store the tokens and cleaned_text 
text_desc = df['Ticket Description']
tokens, cleaned_text = preprocess_text(text_description = text_desc)

print(f"First token: {tokens[0]} -- First cleaned text: {cleaned_text[0]}")

First token: ['PRODUCT', 'billing', 'zip', 'code', 'request', 'website', 'address', 'double', 'email', 'address', 'troubleshoot', 'step', 'mention', 'user', 'manual'] -- First cleaned text: i am having an issue with the PRODUCT please assist your billing zip code is we appreciate that you have requested a website address please double check your email address i have tried troubleshooting steps mentioned in the user manual but the issue persists


In [18]:
# Save cleaned_text to a temporary file
cleaned_text.to_csv("temp_cleaned.csv", index=False, header=["cleaned_description"])

cleaned_text_path = os.path.join(root_path, 'data', 'processed', 'cleaned_text')

# Move it to your processed folder
store_dataset(
    source_path="temp_cleaned.csv",
    target_directory=cleaned_text_path,
    target_filename="cleaned_text.csv"
)

Directory 'D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\processed\cleaned_text' created.
Successfully stored: D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\processed\cleaned_text\cleaned_text.csv


In [19]:
# Save tokens to a temporary JSON file
tokens.to_json("temp_tokens.json")

tokens_path = os.path.join(root_path, 'data', 'processed', 'tokens')

# Move it to your processed folder
store_dataset(
    source_path="temp_tokens.json",
    target_directory=tokens_path,
    target_filename="tokens.json"
)

Directory 'D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\processed\tokens' created.
Successfully stored: D:\MSc Data Science\TB 2\IATA\CW\EMATM0067_2025_TB-2-g43\data\processed\tokens\tokens.json


In [20]:
# Build the Bag of words vectors, tf-idf vectors, skipgram embeddings and sbert embeddings 
def build_all_representations(tokens, cleaned_text):
    tokens, sbert_text = tokens, cleaned_text
    clean_text = tokens.apply(lambda t: ' '.join(t))
    print("Building BoW...", end=" ")
    X_bow = vectorize_boa(clean_text)
    print(f"shape: {X_bow.shape}")
    print("Building TF-IDF...", end=" ")
    X_tfidf = vectorize_tfidf(clean_text)
    print(f"shape: {X_tfidf.shape}")
    print("Building Skipgram...", end=" ")
    X_skipgram = vectorize_skipgram(tokens)
    print(f"shape: {X_skipgram.shape}")
    print("Building SBERT...", end=" ")
    X_sbert = vectorize_sbert(sbert_text)
    print(f"shape: {X_sbert.shape}")
    return X_bow, X_tfidf, X_skipgram, X_sbert

In [21]:
# Invoke the methods build the BoW, Tf-idf vectors, skipgram embeddings and sbert embeddings 
X_bow, X_tfidf, X_skipgram, X_sbert = build_all_representations(tokens, cleaned_text)

Building BoW... shape: (8469, 5000)
Building TF-IDF... shape: (8469, 3642)
Building Skipgram... ['PRODUCT', 'billing', 'zip', 'code', 'request', 'website', 'address', 'double', 'email', 'address', 'troubleshoot', 'step', 'mention', 'user', 'manual']
['PRODUCT', 'exist', 'PRODUCT', 'intermittent', 'work', 'time', 'act', 'unexpectedly']
['PRODUCT', 'PRODUCT', 'turn', 'yesterday', 'respond', 'original', 'charger', 'come', 'PRODUCT', 'charge', 'properly']
['PRODUCT', 'interested', 'love', 'happen', 'feedback', 'contact', 'time', 'remain', 'unresolved']
['PRODUCT', 'note', 'seller', 'responsible', 'damage', 'arise', 'delivery', 'battleground', 'game', 'game', 'good', 'condition', 'ship', 'noticed', 'sudden', 'decrease', 'battery', 'life', 'PRODUCT', 'longer']
['PRODUCT', 'PRODUCT', 'turn', 'yesterday', 'respond', 'remove', 'new', 'productpurch', 'check', 'update', 'PRODUCT']
['access', 'PRODUCT', 'keep', 'display', 'invalid', 'credential', 'error', 'correct', 'login', 'regain', 'access', 's

Batches:   0%|          | 0/133 [00:00<?, ?it/s]

shape: (8469, 384)


In [22]:
# Mapping of variable names to their objects
embeddings = {
    "bow_vectors": X_bow,
    "tfidf_vectors": X_tfidf,
    "skipgram_embeddings": X_skipgram,
    "sbert_embeddings": X_sbert
}

# This approach requires checking the type of data
for filename, data in embeddings.items():

    target_path = os.path.join(root_path, 'data', 'processed', filename)
    
    if sp.issparse(data):
        # Save Scipy Sparse Matrix
        sp.save_npz(f"{target_path}.npz", data)
        print(f"Saved sparse matrix: {filename}.npz")
    else:
        # Save Numpy Array
        np.save(f"{target_path}.npy", data)
        print(f"Saved dense array: {filename}.npy")

Saved sparse matrix: bow_vectors.npz
Saved sparse matrix: tfidf_vectors.npz
Saved dense array: skipgram_embeddings.npy
Saved dense array: sbert_embeddings.npy
